# Fine-tuning V1 — Balanceamento pela Mediana

Todas as classes igualadas à mediana do dataset (2.117 amostras):
classes maiores são amostradas para baixo; classes menores são oversampleadas.
5 epochs.

### Configuração de ambiente

In [ ]:
from os import environ

environ['CUDA_VISIBLE_DEVICES'] = input('GPU ID: ')

### Imports

In [ ]:
from os.path import join
from json import load, dump
from datetime import timedelta
from collections import defaultdict
import random

from sklearn.model_selection import StratifiedGroupKFold
from unsloth import FastVisionModel
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm

import torch

from scripts.authentication import authenticate_huggingface
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis
from scripts.messages import create_training_message
from scripts.training import Training

import scripts.definitions as defs

### Autenticação

In [ ]:
authenticate_huggingface()

### Estratégia de balanceamento

Cap = Floor = mediana (2.117): todas as classes ficam com exatamente 2.117 amostras.

In [ ]:
MAX_CAP   = 2117  # mediana do dataset
MIN_FLOOR = 2117  # todas as classes igualadas à mediana

### Carregamento do dataset e divisão treino/teste

In [ ]:
with open(join(defs.DATA_PATH, 'stt_data', 'simple_dataset.json'), 'r', encoding='utf-8') as file:
    full_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

exams  = [entry.exam_id for entry in full_dataset]
labels = [entry.report.skin_lesion for entry in full_dataset]

sgkf = StratifiedGroupKFold(
    n_splits=int(1.0 / defs.TEST_PROPORTION),
    shuffle=True,
    random_state=defs.STATIC_RANDOM_STATE
)

training_indices, _ = next(sgkf.split(full_dataset, labels, exams))
training_data = [full_dataset[i] for i in training_indices]

print(f'Split de treino (antes do balanceamento): {len(training_data)} amostras')

### Balanceamento

In [ ]:
random.seed(defs.STATIC_RANDOM_STATE)

class_groups = defaultdict(list)
for entry in training_data:
    class_groups[entry.report.skin_lesion].append(entry)

balanced = []
for cls, entries in class_groups.items():
    n = len(entries)
    cap    = min(n, MAX_CAP) if MAX_CAP is not None else n
    target = max(cap, MIN_FLOOR)

    if n >= target:
        sampled = random.sample(entries, target)
    else:
        sampled = entries.copy()
        while len(sampled) < target:
            remaining = target - len(sampled)
            sampled += random.sample(entries, min(len(entries), remaining))
        sampled = sampled[:target]

    balanced.extend(sampled)
    print(f'  {cls}: {n} → {len(sampled)}')

random.shuffle(balanced)
training_data = balanced
print(f'\nTotal balanceado: {len(training_data)} amostras')

### Configuração de treinamento

In [ ]:
VERSION = 'V1'

training_hyperparameters = Training(
    base_model_name=defs.BASE_MODEL_NAME,
    trained_model_name=defs.MODEL_NAME,
    quantization=True,
    prompt_type=defs.PromptType.REPORT,
    version=VERSION,
    size=11,
    peft_hyperparameters={
        'finetune_vision_layers': True,
        'finetune_language_layers': True,
        'finetune_attention_modules': True,
        'finetune_mlp_modules': True,
        'r': 128,
        'lora_alpha': 128,
        'lora_dropout': 0.1,
        'bias': 'none',
        'random_state': defs.STATIC_RANDOM_STATE,
        'use_rslora': True,
        'loftq_config': None
    },
    sft_hyperparameters={
        'per_device_train_batch_size': 4,
        'gradient_accumulation_steps': 1,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'num_train_epochs': 5.0,
        'lr_scheduler_type': 'cosine',
        'warmup_ratio': 0.1,
        'optim': 'paged_adamw_32bit',
        'logging_steps': 1,
        'report_to': 'tensorboard',
        'output_dir': 'outputs',
        'seed': defs.STATIC_RANDOM_STATE,
        'bf16': is_bf16_supported(),
        'fp16': not is_bf16_supported(),
        'remove_unused_columns': False,
        'dataset_text_field': '',
        'dataset_kwargs': {'skip_prepare_dataset': True},
        'dataset_num_proc': 4,
        'max_seq_length': defs.MAX_TOKENS
    },
    used_memory=0.0,
    training_time=0.0
)

with open(join(defs.TRAINING_PATH, f'hyperparameters_{VERSION}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

### Preparação das mensagens

In [ ]:
training_messages = []

for lesion_data in tqdm(training_data, desc='Criando mensagens: '):
    training_messages.append(create_training_message(
        training_hyperparameters.prompt_type,
        lesion_data,
        training_dataset_analysis
    ))

### Inicialização do LLaMA 3.2

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    training_hyperparameters.base_model_name,
    load_in_4bit=training_hyperparameters.quantization,
    use_gradient_checkpointing='unsloth'
)

### Configuração do treinador

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    **training_hyperparameters.peft_hyperparameters
)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_messages,
    args=SFTConfig(**training_hyperparameters.sft_hyperparameters),
)

### Treinamento

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f'Tempo de treinamento: {timedelta(seconds=trainer_stats.metrics["train_runtime"])}')
print(f'Memória máxima reservada: {used_memory} GB')

training_hyperparameters.used_memory = used_memory
training_hyperparameters.training_time = trainer_stats.metrics['train_runtime']

hyperparameters_name = f'hyperparameters_{VERSION}'
if training_hyperparameters.quantization:
    hyperparameters_name += '-4bit'

with open(join(defs.TRAINING_PATH, f'{hyperparameters_name}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

### Salvamento

In [ ]:
trained_model_name = f'{training_hyperparameters.trained_model_name}-{VERSION}-{training_hyperparameters.size}B'
if training_hyperparameters.quantization:
    trained_model_name += '-4bit'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
    models = {name: defs.Model(**data) for name, data in __import__('json').loads(content).items()} if content else {}

models[trained_model_name] = defs.Model(
    local=True,
    quantized=training_hyperparameters.quantization,
    prompt_type=training_hyperparameters.prompt_type,
    version=VERSION,
    size=training_hyperparameters.size
)

for name, m in models.items():
    models[name] = m.model_dump()

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print(f'Modelo salvo em: {save_path}')